# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Ejecutar esta Celda Si se ejecuta de forma local con Jupyter, si es colab Ejecuitar la siguiente Celda
import os
from groq import Groq

# Antes de ejecutar, define la variable de entorno en tu sistema:
# En Linux/Mac:
# export GROQ_API_KEY="tu_api_key"
#
# En Windows (PowerShell):
# setx GROQ_API_KEY "tu_api_key"

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Cliente de Groq inicializado correctamente.")


Cliente de Groq inicializado correctamente.


In [ ]:
# No ejecitar si ya ejecutaste la anterior
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
from groq import Groq
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [2]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [4]:
# Verificamos los modelos disponibles en mi cuenta Groq  
models = client.models.list()
for m in models.data:
    print(m.id)


canopylabs/orpheus-arabic-saudi
whisper-large-v3
openai/gpt-oss-120b
groq/compound
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-20b
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo


In [5]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro / SSD)**  

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Tipo de memoria** | Volátil: pierde su contenido al apagar o reiniciar el equipo. | No volátil: conserva los datos aunque el equipo esté apagado. |
| **Función principal** | Guarda temporalmente los datos y programas que la CPU está usando en ese momento. Facilita el acceso rápido a la información que se está procesando. | Guarda de forma permanente archivos, sistemas operativos, aplicaciones, fotos, videos, etc. |
| **Velocidad** | Mucho más rápida (nanosegundos). Permite que la CPU lea/escriba datos casi al instante. | Más lenta (micro‑ a milisegundos). Los SSD son mucho más rápidos que los HDD tradicionales, pero siguen siendo más lentos que la RAM. |
| **Capacidad típica** | Entre 4 GB y 128 GB en la mayoría de los PCs de consumo. | Desde 128 GB (SSD) hasta varios terabytes (HDD o SSD). |
| **Costo por GB** | Relativa

In [6]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "id": "chatcmpl-83d09680-4499-4628-ba1c-6793ced0b5cb",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro / SSD)**  \n\n| Característica | RAM | Almacenamiento |\n|----------------|-----|----------------|\n| **Tipo de memoria** | Volátil: pierde su contenido al apagar o reiniciar el equipo. | No volátil: conserva los datos aunque el equipo esté apagado. |\n| **Función principal** | Guarda temporalmente los datos y programas que la CPU está usando en ese momento. Facilita el acceso rápido a la información que se está procesando. | Guarda de forma permanente archivos, sistemas operativos, aplicaciones, fotos, videos, etc. |\n| **Velocidad** | Mucho más rápida (nanosegundos). Permite que la CPU lea/escriba datos casi al instante. | Más lenta (micro‑ a milisegundos). Los SSD son mucho más rápidos que los HDD tradicionales, pero siguen sien

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [7]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 86
Tokens de la respuesta: 644
Tokens totales: 730


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [12]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
# Estyo usando meta-llama/llama-prompt-guard-2-22m  porque el modelo llama-3.1-8b-instant
# Solo esta disponible para la version enterprise

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="meta-llama/llama-prompt-guard-2-22m",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 0.42 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [14]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
# Cambio de modelo llama-3.3-70b-versatile  por el whisper-large-v3-turbo

inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 0.42 s — 730 tokens
Modelo grande: 0.96 s — 698 tokens

Respuesta del modelo grande:
 **RAM (Memoria de Acceso Aleatorio)**  
- **Volátil:** se borra cuando la computadora se apaga.  
- **Velocidad:** mucho más rápida que cualquier tipo de almacenamiento.  
- **Función:** guarda de forma temporal los datos e instrucciones que el CPU necesita usar en ese momento (por ejemplo, el programa que estás ejecutando, las páginas abiertas del navegador, el sistema operativo en ejecución).  
- **Capacidad:** suele ser menor que el almacenamiento (8 GB, 16 GB, 32 GB, etc.).  
- **Tipo de acceso:** aleatorio, es decir, el CPU puede leer o escribir en cualquier dirección casi instantáneamente.

**Almacenamiento (Disco duro, SSD, etc.)**  
- **No volátil:** conserva los datos incluso cuando la computadora está apagada.  
- **Velocidad:** más lento que la RAM (especialmente los discos duros mecánicos). Los SSD son más rápidos pero aún más lentos que la RAM.  
- **Función:** guarda todo 